In [1]:
import os
import json
from dotenv import load_dotenv
from utils import set_seed, generate_namespace

from lora_fine_tune.model_local import LocalLM
from lora_fine_tune.lora_utils import apply_lora
from lora_fine_tune.data_utils import build_dataset

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
load_dotenv()
api_key = os.environ["OPENAI_API_KEY"]

cfg = generate_namespace(path=f"../config.yaml")
print(json.dumps(vars(cfg), indent=2))

set_seed(cfg.seed)

{
  "model_name": "gpt2",
  "api_model_name": "gpt-4.1-nano",
  "seed": 42,
  "max_token_length": 128
}


In [3]:
model = LocalLM(cfg.model_name)

total_params = sum(p.numel() for p in model.model.parameters())
trainable_params = sum(p.numel() for p in model.model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 124,439,808
Trainable parameters: 124,439,808


In [4]:
lora_model = apply_lora(model.model)
lora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/opt/miniconda3/envs/lora-fine-tune-env/lib/python3.12/site-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [5]:
dataset = build_dataset()
tokenized = model.tokenize_dataset(dataset, cfg.max_token_length, padding=False)
tokenized

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 3
})